In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
import polars as pl
import pandas as pd

# Assuming 'df_full' is your original unsplit dataset
df_full = pl.read_csv("/home/biostat/Register_data/data_request/RA/Stratafit/05_data_ready_to_use/mockClientData/realSplitData/*.csv").to_pandas()

columns = ['DAS28', 'CRP', 'ESR', 'SJC28', 'TJC28']

central_imputer = IterativeImputer(estimator=BayesianRidge(), max_iter=20, random_state=42, imputation_order="roman")
df_central_imputed = pd.DataFrame(
    central_imputer.fit_transform(df_full[columns]), 
    columns=columns
)

In [27]:
import pandas as pd
import numpy as np
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

def apply_federated_mice(df: pd.DataFrame, model_config: dict):
    """
    Applies the federated MICE parameters to a new/full dataframe.
    """
    # 1. Extract info from the Vantage6 result
    state = model_config['state']
    columns = model_config['parameters']['columns']
    global_estimates = state['global_estimates']
    
    # Ensure we only work with the columns the model was trained on
    data = df[columns].copy().values
    
    # 2. Create a 'blank' imputer
    # We set max_iter=1 because we only need the imputer to perform 
    # the transformation logic once using our injected weights.
    imputer = IterativeImputer(estimator=BayesianRidge(), max_iter=1, random_state=42)
    
    # 3. 'Prime' the imputer
    # It needs to 'fit' once just to build the internal imputation_sequence_ 
    # (the list of regression models for each column).
    imputer.fit(data)
    
    # 4. Surgical Injection of Federated Weights
    for i, est_data in enumerate(global_estimates):
        # Access the internal regression model for this step in the chain
        # Using index [2] to get the estimator object
        target_estimator = imputer.imputation_sequence_[i][2]
        
        # Manually set the learned coefficients and intercept
        target_estimator.coef_ = np.array(est_data['coef'])
        target_estimator.intercept_ = est_data['intercept']
        
    # 5. Transform (the actual imputation)
    imputed_values = imputer.transform(data)
    
    # 6. Return as a nice DataFrame
    return pd.DataFrame(imputed_values, columns=columns, index=df.index)

# --- EXAMPLE USAGE ---
# vantage_output = [{'schema_version': 1, ...}] # Your JSON result
# df_full = pd.read_csv("your_data.csv")

# df_imputed = apply_federated_mice(df_full, vantage_output[0])
# print(df_imputed.head())

In [28]:
model_config = {'schema_version': 1, 'type': 'imputation', 'strategy': 'mice', 'fitted': True, 'parameters': {'columns': ['DAS28', 'CRP', 'ESR', 'SJC28', 'TJC28'], 'max_iter': 20}, 'state': {'global_estimates': [{'feat_idx': 0, 'neighbor_indices': [1, 2, 3, 4], 'coef': [-0.01672221679632235, 0.027626978159820206, 0.11355008679404215, 0.1518962100977987], 'intercept': 2.028687954117019}, {'feat_idx': 1, 'neighbor_indices': [0, 2, 3, 4], 'coef': [-0.1049888461776609, 0.038592269067623926, 0.11791376441699533, 0.020051684345092094], 'intercept': -0.09701252383389546}, {'feat_idx': 2, 'neighbor_indices': [0, 1, 3, 4], 'coef': [15.02943742945979, 3.9742141523600427, -1.5078058637543916, -2.30123844483662], 'intercept': -19.062676693614222}, {'feat_idx': 3, 'neighbor_indices': [0, 1, 2, 4], 'coef': [2.2380017326960226, 0.3676487953340618, -0.056219301841980124, -0.21446612647143054], 'intercept': -3.8085784300443275}, {'feat_idx': 4, 'neighbor_indices': [0, 1, 2, 3], 'coef': [4.793665264289738, 0.25831164975927934, -0.13630497502962668, -0.3440981996096917], 'intercept': -9.150212793275038}]}, 'metadata': {'created_at': '2026-03-10T11:47:05.578768+00:00Z', 'n_organizations': 3}}

In [29]:
# Assuming 'central_imputer' is your fitted sklearn IterativeImputer
# and 'columns' is your list ['DAS28', 'CRP', 'ESR', 'SJC28', 'TJC28']

central_results = []
for step in central_imputer.imputation_sequence_:
    feat_idx = step[0]
    neighbor_indices = step[1]
    estimator = step[2]
    
    if int(feat_idx == 2):
        central_results.append({
            "feat_idx": int(feat_idx),
            "column_name": columns[feat_idx],
            "neighbor_indices": neighbor_indices.tolist(),
            "coef": estimator.coef_.tolist(),
            "intercept": float(estimator.intercept_)
        })

# Print them out to compare side-by-side
import json
print(json.dumps(central_results, indent=2))

[
  {
    "feat_idx": 2,
    "column_name": "ESR",
    "neighbor_indices": [
      0,
      1,
      3,
      4
    ],
    "coef": [
      14.916690191534139,
      4.0453469468946555,
      -1.5042763523282534,
      -2.2780225598597523
    ],
    "intercept": -18.832222026220357
  },
  {
    "feat_idx": 2,
    "column_name": "ESR",
    "neighbor_indices": [
      0,
      1,
      3,
      4
    ],
    "coef": [
      14.912091484397042,
      4.0461728886458275,
      -1.502470456199392,
      -2.277607148390759
    ],
    "intercept": -18.821665416228264
  },
  {
    "feat_idx": 2,
    "column_name": "ESR",
    "neighbor_indices": [
      0,
      1,
      3,
      4
    ],
    "coef": [
      14.910853323202259,
      4.046397593997314,
      -1.5021206139508432,
      -2.2773900900190864
    ],
    "intercept": -18.818882283771437
  },
  {
    "feat_idx": 2,
    "column_name": "ESR",
    "neighbor_indices": [
      0,
      1,
      3,
      4
    ],
    "coef": [
      14.910454

In [30]:
# In your test script
print(central_imputer.imputation_sequence_)

[_ImputerTriplet(feat_idx=np.int64(0), neighbor_feat_idx=array([1, 2, 3, 4]), estimator=BayesianRidge()), _ImputerTriplet(feat_idx=np.int64(1), neighbor_feat_idx=array([0, 2, 3, 4]), estimator=BayesianRidge()), _ImputerTriplet(feat_idx=np.int64(2), neighbor_feat_idx=array([0, 1, 3, 4]), estimator=BayesianRidge()), _ImputerTriplet(feat_idx=np.int64(3), neighbor_feat_idx=array([0, 1, 2, 4]), estimator=BayesianRidge()), _ImputerTriplet(feat_idx=np.int64(4), neighbor_feat_idx=array([0, 1, 2, 3]), estimator=BayesianRidge()), _ImputerTriplet(feat_idx=np.int64(0), neighbor_feat_idx=array([1, 2, 3, 4]), estimator=BayesianRidge()), _ImputerTriplet(feat_idx=np.int64(1), neighbor_feat_idx=array([0, 2, 3, 4]), estimator=BayesianRidge()), _ImputerTriplet(feat_idx=np.int64(2), neighbor_feat_idx=array([0, 1, 3, 4]), estimator=BayesianRidge()), _ImputerTriplet(feat_idx=np.int64(3), neighbor_feat_idx=array([0, 1, 2, 4]), estimator=BayesianRidge()), _ImputerTriplet(feat_idx=np.int64(4), neighbor_feat_id

In [31]:
df_fed_combined = apply_federated_mice(df=df_full, model_config=model_config)

/home/josef/stratafit_repos/stratafit-org-github/strata-fit-v6-imputation-py/.venv/lib/python3.12/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [32]:
import numpy as np

# Combine your imputed node dataframes into one: df_fed_combined
diff = df_central_imputed - df_fed_combined
mse = np.mean(diff.values**2)

print(f"Federated vs Centralized MSE: {mse}")

Federated vs Centralized MSE: 4688.95822573879


In [25]:
for col in columns:
    col_mse = np.mean((df_central_imputed[col] - df_fed_combined[col])**2)
    print(f"MSE for {col}: {col_mse}")

MSE for DAS28: 0.04001958630380663
MSE for CRP: 0.0008024276332764123
MSE for ESR: 52.77172838020929
MSE for SJC28: 0.02021433032577633
MSE for TJC28: 0.010926252407910451


In [ ]:
pl.from_pandas(df_full).select("ESR").describe()